# Finetune BERT (bert-base-uncased) for Next Sentence Prediction
**Solhee Tucker**  
May 2026  
Final project for ML in Bioinformatics class, 2026 spring, San José State University  
Instructor: Dr. Williams (Bill) Andreopoulos

### GPU Setup

In [ ]:
# === GPU / device setup (works in Google Colab and local) ===
import sys, platform, torch

IN_COLAB = "google.colab" in sys.modules

print(f"Running in Google Colab : {IN_COLAB}")
print(f"Platform               : {platform.platform()}")
print(f"Python                 : {sys.version.split()[0]}")
print(f"PyTorch                : {torch.__version__}")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA available         : True")
    print(f"CUDA device name       : {torch.cuda.get_device_name(0)}")
    print(f"CUDA device count      : {torch.cuda.device_count()}")
    print(f"CUDA version (torch)   : {torch.version.cuda}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon GPU (MPS)")
else:
    device = torch.device("cpu")
    print("No GPU detected -- falling back to CPU")

print(f"\n>>> Using device: {device}")


### Library Imports

We will use HuggingFace `transformers` (`BertTokenizer`, `BertForNextSentencePrediction`), PyTorch for training, and `scikit-learn` for accuracy. Seeds are fixed for reproducibility.

In [ ]:
# If running in Colab for the first time, uncomment to install:
# !pip install -q transformers scikit-learn pandas matplotlib tqdm

import os
import random
import numpy as np
import pandas as pd

# PyTorch
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# HuggingFace Transformers
import transformers
from transformers import (
    BertTokenizer,
    BertForNextSentencePrediction,
    get_linear_schedule_with_warmup,
)

# Metrics & utilities
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

pd.set_option("display.max_colwidth", 120)
print("transformers :", transformers.__version__)
print("pandas       :", pd.__version__)
print("numpy        :", np.__version__)
print("All imports OK")


### Load Datasets

The CSVs have **no header**, contain trailing empty columns (artifact of Excel export), and are encoded in `latin-1` (not UTF-8). We handle all three when loading.

**Column convention**: `sentence_a`, `sentence_b`, `label`  
**Label convention**: `0` = IsNextSentence (positive), `1` = NotNextSentence (negative).

In [12]:
from pathlib import Path

DATA_DIR = Path("../content")


def load_pairs(filename: str) -> pd.DataFrame:
    path = DATA_DIR / filename
    df = pd.read_csv(
        path,
        header=None,
        usecols=[0, 1, 2],
        names=["sentence_a", "sentence_b", "label"],
        encoding="latin-1",
    )
    df["label"] = df["label"].astype(int)
    return df


test_df = load_pairs("TestSet-2.csv")
train_df = load_pairs("trainingdata-2.csv")
val_df = load_pairs("validationdata-2.csv")

print(">>> Dataset shapes (before deduplication)")
print(f"train : {train_df.shape}")
print(f"val   : {val_df.shape}")
print(f"test  : {test_df.shape}")

train_df = train_df.drop_duplicates().reset_index(drop=True)
val_df   = val_df.drop_duplicates().reset_index(drop=True)
test_df  = test_df.drop_duplicates().reset_index(drop=True)

print(">>> Dataset shapes (after deduplication)")
print(f"train : {train_df.shape}")
print(f"train : {train_df.shape}")
print(f"val   : {val_df.shape}")
print(f"test  : {test_df.shape}")


>>> Dataset shapes (before deduplication)
train : (960, 3)
val   : (137, 3)
test  : (103, 3)
>>> Dataset shapes (after deduplication)
train : (954, 3)
train : (954, 3)
val   : (137, 3)
test  : (103, 3)


In [13]:
display(train_df.head())

,sentence_a,sentence_b,label
0,The Philippines is a country in Southeast Asia with a rich cultural heritage.,"Filipino cuisine includes dishes influenced by Chinese, Spanish, and American cuisine.",0
1,Singapore is a small island nation in Southeast Asia known for its modernity and cleanliness.,"The Merlion, a mythical creature with the head of a lion and the body of a fish, is a symbol of Singapore.",0
2,South Korea is a country in East Asia known for its K-pop music and popular dramas.,"Gangnam Style, a song by the South Korean singer Psy, became a global hit in 2012.",0
3,Sri Lanka is an island nation in South Asia known for its tea plantations and ancient temples.,Colombo is the largest city in Sri Lanka.,0
4,Taiwan is a democratic country in East Asia with a thriving economy.,"Taipei 101, a skyscraper in Taipei, was the tallest building in the world from 2004 to 2010.",0


In [14]:

print("\n>>> Label distribution (0 = IsNext, 1 = NotNext)")
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    counts = df["label"].value_counts().to_dict()
    print(f"  {name:5s}: {counts}  (total={len(df)})")

# Sentence-length statistics
word_counts = pd.concat([
    df[col].str.split().str.len()
    for df in [train_df, val_df, test_df]
    for col in ["sentence_a", "sentence_b"]
])
print(word_counts.describe(percentiles=[0.5, 0.95, 0.99]))



>>> Label distribution (0 = IsNext, 1 = NotNext)
  train: {1: 480, 0: 474}  (total=954)
  val  : {1: 71, 0: 66}  (total=137)
  test : {0: 54, 1: 49}  (total=103)
count    2388.000000
mean       11.405360
std         3.714554
min         6.000000
50%        11.000000
95%        18.000000
99%        21.130000
max        24.000000
dtype: float64


In [15]:
# Source: https://huggingface.co/docs/transformers/v4.22.1/en/model_doc/bert#transformers.BertForNextSentencePrediction.forward.example
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForNextSentencePrediction.from_pretrained("bert-base-uncased")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 10411.56it/s]
[transformers] BertForNextSentencePrediction LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
# Source: https://huggingface.co/docs/transformers/v4.22.1/en/model_doc/bert#transformers.BertForNextSentencePrediction.forward.example
prompt = "In Italy, pizza served in formal settings, such as at a restaurant, is presented unsliced."
next_sentence = "The sky is blue due to the shorter wavelength of blue light."
encoding = tokenizer(prompt, next_sentence, return_tensors="pt")

outputs = model(**encoding, labels=torch.LongTensor([1]))
logits = outputs.logits
assert logits[0, 0] < logits[0, 1]  # next sentence was random

In [ ]:
MAX_LENGTH = 64
